# Multi-Group Zooplankton Simulation

This notebook demonstrates a simulation with:

-   **2 zooplankton groups**: Surface (epipelagic) and Migrant (diel vertical migration)
-   **2 depth layers**: 0m (25°C) and 50m (10°C)
-   **Grid**: 50x50 cells, 1° resolution
-   **Duration**: 90 days, daily timestep
-   **Forcings**:
    -   Temperature: Constant by layer
    -   NPP: Gaussian blob at center (constant in time)
    -   Currents: Circular vortex
    -   Day length: Seasonal variation


## 1. Setup and Imports


In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib import cm

from seapopym_message.core.group import FunctionalGroup, UnitInstance
from seapopym_message.core.kernel import Kernel
from seapopym_message.forcing import ForcingManager, ForcingSource
from seapopym_message.model.zooplankton import zooplankton_group
from seapopym_message.transport.worker import TransportWorker

# Visualization settings
plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

## 2. Grid and Parameters


In [ ]:
# Grid
nlat, nlon = 50, 50
lat = np.linspace(-25, 25, nlat)
lon = np.linspace(-25, 25, nlon)
depth_levels = np.array([0, 50])  # meters
n_depths = len(depth_levels)

# Time
n_days = 30 * 12
time_days = np.arange(n_days)
dt = 1.0  # day

# Biological parameters (shared)
bio_params = {
    "tau_r0": 10.0,
    "gamma_tau_r": 0.1,
    "T_ref": 0.0,
    "lambda_0": 1 / 150,
    "gamma_lambda": 0.1,
    "n_ages": 10,
    "E": 0.15,
}

print(f"Grid: {nlat}x{nlon}, {n_depths} layers")
print(f"Duration: {n_days} days, dt={dt} day")
print(f"Depth levels: {depth_levels} m")

## 3. Create Forcings


In [ ]:
# 3.1 Temperature (3D: time, depth, lat, lon)
# Constant in time, varies by depth
temp_surface = 25.0  # °C
temp_deep = 10.0  # °C

temp_data = np.zeros((n_days, n_depths, nlat, nlon))
temp_data[:, 0, :, :] = temp_surface  # Surface layer
temp_data[:, 1, :, :] = temp_deep  # Deep layer

temp_da = xr.DataArray(
    temp_data,
    dims=("time", "depth", "lat", "lon"),
    coords={
        "time": time_days,
        "depth": depth_levels,
        "lat": lat,
        "lon": lon,
    },
    name="temperature",
)

print(f"Temperature: {temp_da.shape}")
print(f"  Surface: {temp_surface}°C, Deep: {temp_deep}°C")

In [ ]:
# 3.2 Primary Production (2D: time, lat, lon)
# Gaussian blob at center, constant in time
LAT, LON = np.meshgrid(lat, lon, indexing="ij")
center_lat, center_lon = 0.0, 0.0
sigma_npp = 5.0  # degrees (width of blob)
npp_max = 300.0  # mg C / m² / day

dist_sq = (LAT - center_lat) ** 2 + (LON - center_lon) ** 2
npp_2d = npp_max * np.exp(-dist_sq / (2 * sigma_npp**2))

# Repeat for all timesteps
npp_data = np.tile(npp_2d[np.newaxis, :, :], (n_days, 1, 1))

npp_da = xr.DataArray(
    npp_data,
    dims=("time", "lat", "lon"),
    coords={"time": time_days, "lat": lat, "lon": lon},
    name="npp",
)

print(f"NPP: {npp_da.shape}")
print(f"  Max: {npp_max} mg C/m²/day, Blob at ({center_lat}, {center_lon})")

In [ ]:
# 3.3 Circular Currents (2D: time, lat, lon)
# Vortex centered at (0, 0)
u_max = 0.1  # m/s (max velocity)
vortex_radius = 10.0  # degrees

# Distance from center
r = np.sqrt(LAT**2 + LON**2)
# Tangential velocity (clockwise)
theta = np.arctan2(LON, LAT)

# v_tangential = u_max * (r / vortex_radius) * exp(-r / vortex_radius)
v_magnitude = u_max * (r / vortex_radius) * np.exp(-r / vortex_radius)

# Decompose into u (zonal) and v (meridional)
u_2d = -v_magnitude * np.sin(theta)  # Zonal (East-West)
v_2d = v_magnitude * np.cos(theta)  # Meridional (North-South)

# Constant in time
u_data = np.tile(u_2d[np.newaxis, :, :], (n_days, 1, 1))
v_data = np.tile(v_2d[np.newaxis, :, :], (n_days, 1, 1))

u_da = xr.DataArray(
    u_data,
    dims=("time", "lat", "lon"),
    coords={"time": time_days, "lat": lat, "lon": lon},
    name="u_current",
)

v_da = xr.DataArray(
    v_data,
    dims=("time", "lat", "lon"),
    coords={"time": time_days, "lat": lat, "lon": lon},
    name="v_current",
)

print(f"Currents: u={u_da.shape}, v={v_da.shape}")
print(f"  Max velocity: {u_max} m/s, Vortex radius: {vortex_radius}°")

In [ ]:
# 3.4 Day Length (2D: time, lat, lon)
# Simplified: constant across space, varies with time (seasonal)
# Day length = 0.5 + 0.1 * sin(2*pi*t/365)
day_length_1d = 0.5 + 0.1 * np.sin(2 * np.pi * time_days / 365)
day_length_data = np.tile(day_length_1d[:, np.newaxis, np.newaxis], (1, nlat, nlon))

day_length_da = xr.DataArray(
    day_length_data,
    dims=("time", "lat", "lon"),
    coords={"time": time_days, "lat": lat, "lon": lon},
    name="day_length",
)

print(f"Day length: {day_length_da.shape}")
print(f"  Range: [{day_length_1d.min():.2f}, {day_length_1d.max():.2f}]")

## 4. Visualize Forcings


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Temperature - Surface
im0 = axes[0, 0].contourf(lon, lat, temp_da.isel(time=0, depth=0), cmap="RdYlBu_r", levels=20)
axes[0, 0].set_title("Temperature - Surface (0m)")
axes[0, 0].set_xlabel("Longitude")
axes[0, 0].set_ylabel("Latitude")
plt.colorbar(im0, ax=axes[0, 0], label="°C")

# Temperature - Deep
im1 = axes[0, 1].contourf(lon, lat, temp_da.isel(time=0, depth=1), cmap="RdYlBu_r", levels=20)
axes[0, 1].set_title("Temperature - Deep (50m)")
axes[0, 1].set_xlabel("Longitude")
axes[0, 1].set_ylabel("Latitude")
plt.colorbar(im1, ax=axes[0, 1], label="°C")

# NPP
im2 = axes[0, 2].contourf(lon, lat, npp_da.isel(time=0), cmap="Greens", levels=20)
axes[0, 2].set_title("Primary Production (NPP)")
axes[0, 2].set_xlabel("Longitude")
axes[0, 2].set_ylabel("Latitude")
plt.colorbar(im2, ax=axes[0, 2], label="mg C/m²/day")

# Currents (quiver)
skip = 3  # Subsample for visibility
axes[1, 0].quiver(
    lon[::skip],
    lat[::skip],
    u_da.isel(time=0)[::skip, ::skip],
    v_da.isel(time=0)[::skip, ::skip],
    scale=2.0,
)
axes[1, 0].set_title("Currents (Circular Vortex)")
axes[1, 0].set_xlabel("Longitude")
axes[1, 0].set_ylabel("Latitude")
axes[1, 0].set_aspect("equal")

# Day Length (time series)
axes[1, 1].plot(time_days, day_length_1d)
axes[1, 1].set_title("Day Length Evolution")
axes[1, 1].set_xlabel("Day")
axes[1, 1].set_ylabel("Day Fraction")
axes[1, 1].grid(True)

# Current speed
current_speed = np.sqrt(u_da.isel(time=0) ** 2 + v_da.isel(time=0) ** 2)
im5 = axes[1, 2].contourf(lon, lat, current_speed, cmap="Blues", levels=20)
axes[1, 2].set_title("Current Speed")
axes[1, 2].set_xlabel("Longitude")
axes[1, 2].set_ylabel("Latitude")
plt.colorbar(im5, ax=axes[1, 2], label="m/s")

plt.tight_layout()
plt.show()

## 5. Create ForcingManager


In [ ]:
# Wrap DataArrays in ForcingSource
forcings = [
    ForcingSource(temp_da, name="forcing/temperature_3d"),
    ForcingSource(npp_da, name="forcing/npp"),
    ForcingSource(u_da, name="forcing/u"),
    ForcingSource(v_da, name="forcing/v"),
    ForcingSource(day_length_da, name="forcing/day_length"),
]

forcing_manager = ForcingManager(forcings=forcings)
print(f"ForcingManager created with {len(forcings)} forcings")

## 6. Define Functional Groups


In [ ]:
# Surface Group (epipelagic - stays at surface)
surface_params = bio_params.copy()
surface_params["layer_index"] = 0  # Surface layer

surface_group = zooplankton_group(
    name="surface",
    behavior="epipelagic",
    params=surface_params,
    forcing_map={
        "forcing_3d": "forcing/temperature_3d",
        "npp": "forcing/npp",
    },
)

print("Surface group created")
print(f"  Units: {[u.name if hasattr(u, 'name') else u.unit.name for u in surface_group.units]}")

In [ ]:
# Migrant Group (diel vertical migration)
migrant_params = bio_params.copy()
migrant_params["day_layer_index"] = 1  # Deep during day
migrant_params["night_layer_index"] = 0  # Surface at night

migrant_group = zooplankton_group(
    name="migrant",
    behavior="migrant",
    params=migrant_params,
    forcing_map={
        "forcing_3d": "forcing/temperature_3d",
        "day_length": "forcing/day_length",
        "npp": "forcing/npp",
    },
)

print("Migrant group created")
print(f"  Units: {[u.name if hasattr(u, 'name') else u.unit.name for u in migrant_group.units]}")

## 7. Create Kernel


In [ ]:
kernel = Kernel([surface_group, migrant_group])

print(f"Kernel created with {len(kernel.local_units)} local units")
print("\nUnit names:")
for u in kernel.local_units:
    print(f"  - {u.name}")

print("\nDependency graph:")
print(kernel.visualize_graph())

## 8. Initialize State


In [ ]:
# Initial biomass: Small gaussian blob at center
biomass_init_2d = 0.1 * np.exp(-dist_sq / (2 * sigma_npp**2))

state = {
    # Surface group
    "surface/biomass": jnp.array(biomass_init_2d),
    "surface/production": jnp.zeros((bio_params["n_ages"], nlat, nlon)),
    # Migrant group
    "migrant/biomass": jnp.array(biomass_init_2d),
    "migrant/production": jnp.zeros((bio_params["n_ages"], nlat, nlon)),
}

print("Initial state created")
for key, val in state.items():
    print(f"  {key}: {val.shape}")

In [ ]:
transport_worker = TransportWorker(
    grid_type="spherical",
    lat_min=lat.min(),
    lat_max=lat.max(),
    lon_min=lon.min(),
    lon_max=lon.max(),
    nlat=nlat,
    nlon=nlon,
    lat_bc="closed",
    lon_bc="periodic",
)
D_horizontal = 1000.0  # m²/s

## 9. Run Simulation


In [ ]:
# Storage for time series
history = {
    "time": [],
    "surface_biomass": [],
    "migrant_biomass": [],
    "surface_total": [],
    "migrant_total": [],
}

# Merge all params (both groups have same values here)
global_params = surface_params.copy()
global_params.update(migrant_params)

print("Starting simulation...")

for day in range(n_days):
    # Get forcings for this timestep (as xarray for Sensing Units)
    forcings_xr = forcing_manager.prepare_timestep_xarray(time=float(day))

    # Execute biology (local phase)
    state = kernel.execute_local_phase(
        state=state,
        dt=dt,
        params=global_params,
        forcings=forcings_xr,
    )

    # Store results
    history["time"].append(day)
    history["surface_biomass"].append(np.array(state["surface/biomass"]))
    history["migrant_biomass"].append(np.array(state["migrant/biomass"]))
    history["surface_total"].append(float(jnp.sum(state["surface/biomass"])))
    history["migrant_total"].append(float(jnp.sum(state["migrant/biomass"])))

    if (day + 1) % 10 == 0:
        print(
            f"Day {day + 1}/{n_days}: Surface total={history['surface_total'][-1]:.2f}, "
            f"Migrant total={history['migrant_total'][-1]:.2f}"
        )

print("\nSimulation complete!")

## 10. Visualize Results


In [ ]:
# 10.1 Total Biomass Time Series
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(history["time"], history["surface_total"], label="Surface Group", linewidth=2)
ax.plot(history["time"], history["migrant_total"], label="Migrant Group", linewidth=2)

ax.set_xlabel("Day", fontsize=12)
ax.set_ylabel("Total Biomass (kg/m²)", fontsize=12)
ax.set_title("Total Biomass Evolution", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 10.2 Spatial Distribution (Initial, Middle, Final)
timesteps = [0, n_days // 2, n_days - 1]
fig, axes = plt.subplots(3, 2, figsize=(14, 16))

for i, t_idx in enumerate(timesteps):
    # Surface group
    im0 = axes[i, 0].contourf(lon, lat, history["surface_biomass"][t_idx], levels=20, cmap="YlOrRd")
    axes[i, 0].set_title(f"Surface Group - Day {t_idx}", fontsize=12)
    axes[i, 0].set_xlabel("Longitude")
    axes[i, 0].set_ylabel("Latitude")
    plt.colorbar(im0, ax=axes[i, 0], label="Biomass (kg/m²)")

    # Migrant group
    im1 = axes[i, 1].contourf(lon, lat, history["migrant_biomass"][t_idx], levels=20, cmap="YlGnBu")
    axes[i, 1].set_title(f"Migrant Group - Day {t_idx}", fontsize=12)
    axes[i, 1].set_xlabel("Longitude")
    axes[i, 1].set_ylabel("Latitude")
    plt.colorbar(im1, ax=axes[i, 1], label="Biomass (kg/m²)")

plt.tight_layout()
plt.show()

In [ ]:
# 10.3 Difference Between Groups (Final State)
final_diff = history["surface_biomass"][-1] - history["migrant_biomass"][-1]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.contourf(lon, lat, final_diff, levels=20, cmap="RdBu_r")
ax.set_title("Biomass Difference (Surface - Migrant) at Day 90", fontsize=14, fontweight="bold")
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)
plt.colorbar(im, ax=ax, label="Δ Biomass (kg/m²)")
plt.tight_layout()
plt.show()

In [ ]:
# 10.4 Summary Statistics
print("=" * 60)
print("SIMULATION SUMMARY")
print("=" * 60)
print(f"\nInitial Conditions:")
print(f"  Surface total biomass: {history['surface_total'][0]:.4f} kg/m²")
print(f"  Migrant total biomass: {history['migrant_total'][0]:.4f} kg/m²")

print(f"\nFinal Conditions (Day {n_days}):")
print(f"  Surface total biomass: {history['surface_total'][-1]:.4f} kg/m²")
print(f"  Migrant total biomass: {history['migrant_total'][-1]:.4f} kg/m²")

print(f"\nGrowth Factors:")
surface_growth = history["surface_total"][-1] / history["surface_total"][0]
migrant_growth = history["migrant_total"][-1] / history["migrant_total"][0]
print(f"  Surface: {surface_growth:.2f}x")
print(f"  Migrant: {migrant_growth:.2f}x")

print(
    f"\nMigrant experiences warmer avg temperature (25°C * {0.5:.2f} + 10°C * {0.5:.2f} = {17.5:.1f}°C)"
)
print(f"Surface experiences constant temperature (25°C)")
print(f"\nExpected: Surface should grow faster due to warmer temperature")
print(
    f"Observed: {'✓ Surface > Migrant' if surface_growth > migrant_growth else '✗ Unexpected result'}"
)
print("=" * 60)